# fMRI · 실제 공개 데이터로 확인하기

합성 데이터 예제(`fmri_synthetic_example.ipynb`)에서 연습한 분석을 **실제 사람의 fMRI**에 적용합니다.

- 데이터: OpenNeuro ds000228. 3–12세 어린이와 성인이 짧은 애니메이션 영화를 보는 동안 찍은 fMRI입니다. nilearn이 fMRIPrep으로 전처리하고 4 mm 복셀로 줄인 판을 내려받습니다. 머리 움직임 표(confounds)가 함께 들어 있습니다.
- 이용 조건: 원본(OpenNeuro ds000228)은 CC0이지만, 이 노트북이 받는 nilearn 전처리본과 Harvard-Oxford 아틀라스는 비상업 목적으로만 쓸 수 있습니다. 사용하면 Richardson et al. (2018), *Nature Communications* 9:1027과 아틀라스 논문을 인용합니다. 자세한 내용은 챌린지 안내의 "실제 공개 데이터로 확인하기" 절에 있습니다.
- 합성 데이터와 달리 **정답이 없습니다.** 어느 영역이 실제로 연결되어 있는지 아무도 모릅니다.

처음 실행하면 데이터를 내려받습니다. 데이터는 `DATA_DIR` 폴더에 저장되고, 다음 실행부터는 다시 받지 않습니다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from nilearn import datasets, image, plotting
from nilearn.maskers import NiftiLabelsMasker

DATA_DIR = Path("data_real")   # 데이터를 받을 폴더. 바꾸려면 이 줄만 고칩니다.
N_SUBJECTS = 4                 # 받을 사람 수. 늘리면 용량과 시간이 늘어납니다.
THRESHOLD = 0.5                # FD 기준(mm). 합성 예제와 같습니다.

## 1. 데이터 받기

`fetch_development_fmri`는 사람마다 전처리된 4D 영상(`func`)과 머리 움직임 등 잡음 후보 표(`confounds`)를 받습니다. `phenotypic`에는 나이와 어린이/성인 구분이 들어 있습니다. 사람 수가 3명 이상이면 어린이와 성인이 전체 표본과 비슷한 비율로 섞여 나옵니다.

**주의:** 이 파일들의 머리말(header)에는 TR이 1초로 적혀 있지만 실제 촬영 간격은 2초입니다(원본 데이터의 `task-pixar_bold.json`과 nilearn 문서 기준). 합성 예제처럼 머리말에서 TR을 읽으면 느린 신호를 거르는 기준이 틀어집니다. 실제 데이터에서는 파일에 적힌 값도 확인해야 합니다.

In [ ]:
data = datasets.fetch_development_fmri(n_subjects=N_SUBJECTS, data_dir=str(DATA_DIR))
pheno = pd.DataFrame(data.phenotypic)
subjects = pheno["participant_id"].tolist()
bold0 = nib.load(data.func[0])
tr = 2.0  # 촬영 간격(초). 원본의 task-pixar_bold.json과 nilearn 문서에 적힌 값입니다.
print("사람 수:", len(data.func))
print("볼륨 크기 (x, y, z, 시간):", bold0.shape)
print("파일 머리말에 적힌 TR:", float(bold0.header.get_zooms()[3]), "| 실제 TR:", tr, "초")
pheno[["participant_id", "Age", "Child_Adult", "Gender"]]

## 2. 영역 고르기

합성 데이터의 A·B·C 대신, 해부학적 이름이 붙은 영역 세 개를 고릅니다. Harvard-Oxford 아틀라스(뇌를 이름 붙은 구역으로 나눈 지도)에서 다음을 씁니다.

- **A = 왼쪽 중심앞이랑(precentral gyrus)**: 몸 오른쪽을 움직이는 운동 영역
- **B = 오른쪽 중심앞이랑**: 몸 왼쪽을 움직이는 운동 영역
- **C = 왼쪽 새발톱고랑 안쪽 피질(intracalcarine cortex)**: 뒤통수의 일차 시각 영역

좌우 운동 영역은 쉬는 동안에도 함께 오르내린다고 알려져 있습니다. 시각 영역은 영화를 보는 동안 화면에 따라 반응합니다.

In [ ]:
atlas = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm", symmetric_split=True,
                                            data_dir=str(DATA_DIR))
ROI_NAMES = {"A": "Left Precentral Gyrus",
             "B": "Right Precentral Gyrus",
             "C": "Left Intracalcarine Cortex"}
atlas_img = image.load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
roi_data = np.zeros(atlas_data.shape, dtype=np.int16)
for k, name in enumerate(ROI_NAMES.values(), start=1):
    roi_data[atlas_data == list(atlas.labels).index(name)] = k
rois = image.new_img_like(atlas_img, roi_data)
print({key: int((roi_data == k).sum()) for k, key in enumerate(ROI_NAMES, start=1)}, "(영역별 2 mm 복셀 수)")
plotting.plot_roi(rois, display_mode="ortho", cut_coords=(-10, -20, 30),
                  title="A, B = precentral (L, R) / C = intracalcarine (L)");

## 3. 사람마다 시계열 뽑기

합성 예제와 같은 설정(`detrend`, `high_pass=0.01`, z 점수)으로 영역마다 시계열 하나를 만듭니다. 아틀라스는 2 mm, 영상은 4 mm이므로 masker가 영역 지도를 영상 격자에 맞춰 줍니다.

머리 움직임은 confounds 표의 `framewise_displacement` 열(FD, mm)에서 읽습니다. 첫 시점은 비교할 앞 시점이 없어 값이 비어 있으므로 0으로 둡니다.

In [ ]:
masker = NiftiLabelsMasker(rois, detrend=True, high_pass=0.01, t_r=tr,
                           standardize="zscore_sample")
ts_all, fd_all = {}, {}
for sub, func, conf in zip(subjects, data.func, data.confounds):
    ts_all[sub] = masker.fit_transform(func)            # (시간, 영역)
    fd_all[sub] = pd.read_csv(conf, sep="\t")["framewise_displacement"].fillna(0).to_numpy()
print({sub: ts.shape for sub, ts in ts_all.items()})

## 4. 사람마다 머리 움직임 보기

빨간 점선이 기준(0.5 mm)입니다. 사람마다 움직임의 양이 얼마나 다른지 봅니다.

In [ ]:
fig, axes = plt.subplots(len(subjects), 1, figsize=(12, 1.8 * len(subjects)), sharex=True, sharey=True)
for ax, sub in zip(np.atleast_1d(axes), subjects):
    fd = fd_all[sub]
    group = pheno.set_index("participant_id").loc[sub, "Child_Adult"]
    ax.plot(np.arange(len(fd)) * tr, fd, color="k")
    ax.axhline(THRESHOLD, color="r", ls="--")
    ax.set_ylabel("FD (mm)"); ax.set_title(f"{sub} ({group})", fontsize=9)
ax.set_xlabel("time (s)")
plt.tight_layout()

## 5. 움직임 제거 전후의 상관계수

합성 예제와 같은 규칙으로 FD가 기준보다 큰 시점과 그 바로 다음 시점을 빼고 다시 계산합니다.

In [ ]:
def pairwise(ts):
    c = np.corrcoef(ts.T)
    return {"A-B": c[0, 1], "A-C": c[0, 2], "B-C": c[1, 2]}

rows = []
for sub in subjects:
    ts, fd = ts_all[sub], fd_all[sub]
    keep = fd <= THRESHOLD
    keep[1:] &= fd[:-1] <= THRESHOLD
    before, after = pairwise(ts), pairwise(ts[keep])
    row = {"사람": sub,
           "구분": pheno.set_index("participant_id").loc[sub, "Child_Adult"],
           "평균 FD": fd.mean(),
           "FD>기준 시점": int((fd > THRESHOLD).sum()),
           "남은 시점": f"{keep.sum()}/{len(keep)}"}
    for pair in before:
        row[f"{pair} 전"] = before[pair]
        row[f"{pair} 후"] = after[pair]
    rows.append(row)
result = pd.DataFrame(rows).set_index("사람").round(2)
result

## 설명해 보세요

1. 움직임이 많은 사람과 적은 사람 중 누구의 상관계수가 움직임 제거 전후로 더 크게 바뀌었나요?
2. 이 데이터에는 정답 파일이 없습니다. 그렇다면 어떤 결과를 더 믿을 수 있을까요? 여러 사람에게서 같은 방향으로 나오는 결과, 움직임을 빼도 크게 변하지 않는 결과, 이미 알려진 사실(좌우 운동 영역은 함께 움직인다)과 맞는 결과를 떠올려 보세요.
3. 이 노트북은 머리 움직임 신호를 시계열에서 빼는 처리(confounds 회귀)를 하지 않았습니다. 시점을 빼는 것만으로 움직임의 영향이 모두 사라졌다고 말할 수 있을까요?

**더 해 보기:** `N_SUBJECTS`를 늘리거나 `THRESHOLD`를 0.2나 1.0으로 바꿔 보세요. 사람 수를 늘리면 내려받는 데이터도 늘어납니다.